# 03 Inference

This notebook completes the required inferential analysis:
- one confidence interval and one one-sample test for the selected **population proportion**
- one confidence interval and one one-sample test for the selected **population mean**

The notebook ends with a short final synthesis.


In [1]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

plt.rcParams['figure.dpi'] = 120

ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_selected_cleaned.csv"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"
REF_DIR = ROOT / "references"

for p in [FIG_DIR, TAB_DIR, SUM_DIR, REF_DIR]:
    p.mkdir(parents=True, exist_ok=True)

BEHAVIOR_VAR = "EverCigaretteUse"
BEHAVIOR_P0 = 0.50
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

def ensure_processed():
    raw = pd.read_csv(RAW_PATH)
    behavior_raw = raw[BEHAVIOR_VAR]
    behavior_clean = behavior_raw.where(behavior_raw.isin([1, 2]))
    behavior_binary = pd.Series(
        np.where(behavior_clean == 1, 1, np.where(behavior_clean == 2, 0, np.nan)),
        name="behavior_binary"
    )
    cont_clean = pd.to_numeric(raw[CONT_VAR], errors="coerce")
    cont_clean = cont_clean.where(cont_clean > 0)
    processed = pd.DataFrame({
        "WhatIsYourSex": raw["WhatIsYourSex"],
        BEHAVIOR_VAR: behavior_raw,
        "behavior_binary": behavior_binary,
        CONT_VAR: cont_clean,
    })
    processed.to_csv(PROCESSED_PATH, index=False)
    return raw, processed

raw, processed = ensure_processed()
print("Raw data shape:", raw.shape)
print("Processed data shape:", processed.shape)


Raw data shape: (14041, 103)
Processed data shape: (14041, 4)


## Proportion Inference: `EverCigaretteUse`

In [2]:

behavior_binary = processed["behavior_binary"].dropna().astype(int)
n_b = int(behavior_binary.shape[0])
x_b = int((behavior_binary == 1).sum())
phat = x_b / n_b

ci_b = stats.binomtest(x_b, n_b).proportion_ci(confidence_level=0.95, method="wilson")
z_stat = (phat - BEHAVIOR_P0) / np.sqrt(BEHAVIOR_P0 * (1 - BEHAVIOR_P0) / n_b)
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_stat)))

prop_results = pd.DataFrame({
    "quantity": ["sample_size", "successes", "sample_proportion", "benchmark_p0", "z_statistic", "p_value", "ci_low", "ci_high"],
    "value": [n_b, x_b, phat, BEHAVIOR_P0, z_stat, p_value_b, ci_b.low, ci_b.high]
})
display(prop_results)


,quantity,value
0,sample_size,1.360100e+04
1,successes,7.164000e+03
2,sample_proportion,5.267260e-01
3,benchmark_p0,5.000000e-01
4,z_statistic,6.233744e+00
5,p_value,4.554159e-10
6,ci_low,5.183287e-01
7,ci_high,5.351082e-01


In [3]:
decision_b = "reject H0" if p_value_b < 0.05 else "fail to reject H0"
display(Markdown(f"""**Interpretation (proportion analysis):**  
- The estimated sample proportion is **{phat:.3f}** based on **n = {n_b}** valid students.  
- The benchmark proportion is **p0 = {BEHAVIOR_P0:.2f}**.  
- The 95% confidence interval is **({ci_b.low:.3f}, {ci_b.high:.3f})**.  
- Because the p-value is **{p_value_b:.4g}**, we **{decision_b}** at the 5% level.  
- In context, the proportion of students in the success category for `EverCigaretteUse` appears to be **above 0.50** in this sample.  
- This result is consistent with the EDA, where the success category appeared slightly more common than the failure category.
"""))

**Interpretation (proportion analysis):**  
- The estimated sample proportion is **0.527** based on **n = 13601** valid students.  
- The benchmark proportion is **p0 = 0.50**.  
- The 95% confidence interval is **(0.518, 0.535)**.  
- Because the p-value is **4.554e-10**, we **reject H0** at the 5% level.  
- In context, the proportion of students in the success category for `EverCigaretteUse` appears to be **above 0.50** in this sample.  
- This result is consistent with the EDA, where the success category appeared slightly more common than the failure category.


## Mean Inference: `HowMuchDoYouWeighWithoutShoesInKG`

In [4]:

w = processed[CONT_VAR].dropna()
n_w = int(w.shape[0])
mean_w = float(w.mean())
sd_w = float(w.std(ddof=1))
se_w = sd_w / np.sqrt(n_w)

t_crit = stats.t.ppf(0.975, df=n_w - 1)
ci_low_w = mean_w - t_crit * se_w
ci_high_w = mean_w + t_crit * se_w

t_stat, p_value_w = stats.ttest_1samp(w, popmean=CONT_MU0)

mean_results = pd.DataFrame({
    "quantity": ["sample_size", "sample_mean", "sample_sd", "benchmark_mu0", "t_statistic", "p_value", "ci_low", "ci_high"],
    "value": [n_w, mean_w, sd_w, CONT_MU0, t_stat, p_value_w, ci_low_w, ci_high_w]
})
display(mean_results)


,quantity,value
0,sample_size,13062.000000
1,sample_mean,68.550172
2,sample_sd,16.990868
3,benchmark_mu0,68.000000
4,t_statistic,3.700735
5,p_value,0.000216
6,ci_low,68.258766
7,ci_high,68.841579


In [5]:
decision_w = "reject H0" if p_value_w < 0.05 else "fail to reject H0"
display(Markdown(f"""**Interpretation (mean analysis):**  
- The sample mean weight is **{mean_w:.2f} kg** based on **n = {n_w}** valid students.  
- The sample standard deviation is **{sd_w:.2f} kg**.  
- The benchmark mean is **mu0 = {CONT_MU0:.1f} kg**.  
- The 95% confidence interval for the population mean is **({ci_low_w:.2f}, {ci_high_w:.2f}) kg**.  
- Because the p-value is **{p_value_w:.4g}**, we **{decision_w}** at the 5% level.  
- In context, the average student weight in this sample is **slightly above 68.0 kg**.  
- This result is directionally consistent with the EDA because the sample mean was above the benchmark and the histogram showed a mild right tail.  
- A cautious point is that the difference from 68.0 kg is statistically significant but not very large in practical size.
"""))

**Interpretation (mean analysis):**  
- The sample mean weight is **68.55 kg** based on **n = 13062** valid students.  
- The sample standard deviation is **16.99 kg**.  
- The benchmark mean is **mu0 = 68.0 kg**.  
- The 95% confidence interval for the population mean is **(68.26, 68.84) kg**.  
- Because the p-value is **0.0002159**, we **reject H0** at the 5% level.  
- In context, the average student weight in this sample is **slightly above 68.0 kg**.  
- This result is directionally consistent with the EDA because the sample mean was above the benchmark and the histogram showed a mild right tail.  
- A cautious point is that the difference from 68.0 kg is statistically significant but not very large in practical size.


## Summary Table of Main Inferential Results

In [6]:

inference_summary = pd.DataFrame({
    "analysis": ["Population proportion", "Population mean"],
    "variable": [BEHAVIOR_VAR, CONT_VAR],
    "estimate": [phat, mean_w],
    "benchmark": [BEHAVIOR_P0, CONT_MU0],
    "test_statistic": [z_stat, t_stat],
    "p_value": [p_value_b, p_value_w],
    "ci_low": [ci_b.low, ci_low_w],
    "ci_high": [ci_b.high, ci_high_w],
    "decision_5pct": [decision_b, decision_w]
})
inference_summary.to_csv(TAB_DIR / "03_inference_summary_table.csv", index=False)
display(inference_summary)


,analysis,variable,estimate,benchmark,test_statistic,p_value,ci_low,ci_high,decision_5pct
0,Population proportion,EverCigaretteUse,0.526726,0.5,6.233744,4.554159e-10,0.518329,0.535108,reject H0
1,Population mean,HowMuchDoYouWeighWithoutShoesInKG,68.550172,68.0,3.700735,2.158592e-04,68.258766,68.841579,reject H0


## Final Synthesis

This project asked two question-driven one-sample inference questions using the YRBS 2007 dataset.

For the **behavior variable**, the EDA showed that the success category for `EverCigaretteUse` appeared slightly more common than the failure category. The formal proportion analysis confirmed that pattern: the estimated proportion was above the benchmark of 0.50, the 95% confidence interval stayed above 0.50, and the one-sample z test rejected the null hypothesis.

For the **continuous variable**, the EDA showed a large sample, moderate spread, and a somewhat right-skewed weight distribution with some possible outliers. Even so, the sample mean was slightly above 68.0 kg. The 95% confidence interval and one-sample t test both indicated that the mean weight is statistically different from 68.0 kg, although the practical difference is fairly small.

Overall, the inferential conclusions are consistent with the EDA, and the project workflow supports a clear conclusion from raw data to final interpretation.


In [7]:

final_summary_text = f'''Project Cycle 2 Final Summary

Selected behavior variable: {BEHAVIOR_VAR}
Benchmark proportion: {BEHAVIOR_P0:.2f}
Estimated proportion: {phat:.6f}
95% CI: ({ci_b.low:.6f}, {ci_b.high:.6f})
p-value: {p_value_b:.6g}
Decision at 5%: {decision_b}

Selected continuous variable: {CONT_VAR}
Benchmark mean: {CONT_MU0:.1f}
Estimated mean: {mean_w:.6f}
95% CI: ({ci_low_w:.6f}, {ci_high_w:.6f})
p-value: {p_value_w:.6g}
Decision at 5%: {decision_w}
'''
(SUM_DIR / "final_summary.txt").write_text(final_summary_text, encoding="utf-8")

readme_text = f'''# Project Cycle 2

## Group Information
- Group number: **Replace with your group number**
- Member names: **Replace with member names**

## Dataset
- `YRBS_2007.csv`

## Selected Variables
- Proportion analysis: `{BEHAVIOR_VAR}` with benchmark `p0 = {BEHAVIOR_P0:.2f}`
- Mean analysis: `{CONT_VAR}` with benchmark `mu0 = {CONT_MU0:.1f}`

## Project Questions
1. Is the proportion of students who have ever used cigarettes different from 0.50?
2. Is the mean body weight of students different from 68.0 kg?

## Short Final Conclusion
Using the provided dataset, the sample proportion for `EverCigaretteUse` is above 0.50, and the sample mean for `HowMuchDoYouWeighWithoutShoesInKG` is slightly above 68.0 kg. Both one-sample tests reject the corresponding null hypothesis at the 5% level.
'''
(ROOT / "README.md").write_text(readme_text, encoding="utf-8")
print("Saved:", SUM_DIR / "final_summary.txt")
print("Saved:", ROOT / "README.md")


Saved: /mnt/data/project-cycle-2/outputs/summary/final_summary.txt
Saved: /mnt/data/project-cycle-2/README.md
